# Deepfake Detection Project Walkthrough

This notebook provides a walkthrough of our deepfake detection project, focusing on the steps required to reproduce our final results. It presents a self-contained workflow, distilled from our main development project.

The main project was used for research and development, including experimentation with different model architectures, data augmentation strategies, and hyperparameter tuning. This notebook isolates the final pipeline from that exploratory work, providing a direct method to rerun our experiments and ensure their reproducibility.

The project develops two primary models. The first is a classification model that labels image frames as real, synthetic, or tampered. The second is a segmentation model that identifies manipulated pixels within tampered frames. This guide details the run configurations, data preprocessing steps, and artifact locations necessary to replicate our findings or adapt the workflow to different datasets.

## Quick-start Guide

First, create and activate a new Conda environment with Python 3.10 or newer:
```bash
conda create -n deepfake-env python=3.10
conda activate deepfake-env
```
(If you prefer `venv`, you can create an environment with `python3 -m venv .venv` and activate it accordingly).

Once your environment is active, install the required packages by running the optional installation cell below. Then, launch Jupyter Notebook or JupyterLab and open this file. Work through the cells from top to bottom, and feel free to stop after the classification section if you do not need the segmentation walkthrough.

In [ ]:
# This cell installs the exact libraries and versions needed to reproduce the project results.
# Use this as an alternative to installing packages from your terminal.
%pip install --upgrade pip
%pip install torch==2.8.0 torchvision==0.23.0 datasets scikit-learn matplotlib seaborn pillow albumentations>=1.4.0

### Verify the runtime

Run the next cell to confirm the Python and PyTorch versions that your environment is using and to check whether CUDA hardware is available. Knowing this up front helps set expectations for training speed and avoids confusion when the GPU path is not accessible.

In [ ]:
# The printed summary includes the Python interpreter, the PyTorch build, and the detected GPU name when one is present.
import platform
import torch

print(f"Python version : {platform.python_version()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device    : {torch.cuda.get_device_name(torch.cuda.current_device())}")
else:
    print("Running on CPU. Expect training to take a few extra minutes.")

## Part A — Classification Pipeline
This portion reorganises the classification workflow into clearly defined sections.

### A.1 Initialization Section — Imports And Reproducibility Helpers

In [ ]:
# --- Initialization: Imports And Reproducibility Helpers ---

# Standard library imports for file paths, JSON handling, and basic utilities.
import json
import math
import random
from collections import defaultdict
from datetime import datetime
from pathlib import Path

# Core ML & Data Science libraries for building and training the model.
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch import amp 
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2 as T
import torchvision.utils as vutils

# Libraries for evaluating results and creating plots.
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix


# Sets random seeds across all libraries to ensure runs are reproducible.
def cls_set_seed(seed: int) -> None:
    """Sets the random seed for all relevant libraries to ensure reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Force cuDNN to use deterministic algorithms for reproducibility.
    torch.backends.cudnn.deterministic = True
    # Disable the cuDNN auto-tuner, as it can select non-deterministic algorithms.
    torch.backends.cudnn.benchmark = False


# Generates a unique timestamp, perfect for naming experiment output folders.
def cls_timestamp() -> str:
    """Returns a standardized UTC timestamp string, useful for unique run IDs."""
    return datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')


# Automatically selects the GPU if one is available, otherwise defaults to the CPU.
def cls_device() -> torch.device:
    """Selects and returns the appropriate torch device (GPU if available, else CPU)."""
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### A.2 Configuration Section — Experiment Hyperparameters And Run Directory

Here’s a quick rundown of our configuration and the thinking behind it. To keep our experiments manageable, we resized all our images down. We tested this and found it didn’t result in any real accuracy issues but dramatically sped up our training times, which was a huge win for iterating quickly. For the training process, we started with AdamW because it's a solid, modern optimizer, and then added a cosine scheduler to gradually lower the learning rate, which helped our model converge more effectively. Data augmentation was a real game changer, by randomly transforming the training images, we helped prevent overfitting and made our model better at generalizing. As our list of hyperparameters grew, manually tweaking everything became time consuming, so we used Optuna in our main code to help us tune these knobs and find a strong set of values. Finally, we set a seed to make our results reproducible, for knowing if our changes were actually helping and an output_root to keep all our experiment files neatly organized.

In [ ]:
# --- Classification Configuration Blocks ---

CLASS_CFG = {
    # Dataset window sizes and image resolution (limit pulls from SID_Set + resize for lightweight runs).
    'data': {
        'dataset_name': 'saberzl/SID_Set',      # HF dataset ID supplying Real/Synthetic/Tampered frames.
        'image_size': 128,                      # Final square resolution fed to the CNN.
        'train_samples': 160,                   # Number of training examples materialised per run.
        'val_samples': 64,                      # Validation window to monitor generalisation.
        'test_samples': 64,                     # Held-out slice for quick eval in the notebook.
    },

    # DataLoader orchestration (batching, workers, shuffling) for throughput & determinism.
    'loader': {
        'batch_size': 16,                       # Mini-batch size for all splits.
        'num_workers': 4,                       # Parallel workers per DataLoader.
        'shuffle_train': True,                  # Enable shuffling for training batches.
        'shuffle_eval': False,                  # Keep evaluation order stable for reproducibility.
        'pin_memory': True,                     # Pin host memory to speed GPU transfers.
        'persistent_workers': False,            # Reuse worker processes across epochs when True.
        'prefetch_factor': 2,                   # Number of batches each worker preloads ahead.
    },

    # Core training hyperparameters (optimiser + scheduler bundles) that govern optimisation dynamics.
    'training': {
        'learning_rate': 1e-3,                  # Base LR handed to the optimiser.
        'epochs': 15,                           # Number of passes through the training subset.
        'label_smoothing': 0.05,                # Smooths targets to regularise classification.
        'grad_clip_norm': 2.0,                  # Clips gradient norm to stabilise updates.
        'ema_decay': 0.995,                     # Exponential moving average coefficient for weights.
        'use_amp': True,                        # Toggle PyTorch AMP for faster mixed-precision training.
        'optimizer': {
            'name': 'adamw',                    # Optimiser family (adam/adamw/sgd).
            'weight_decay': 1e-4,               # L2 penalty applied to weights.
            'betas': (0.9, 0.999),              # Adam-style momentum coefficients.
            'momentum': 0.9,                    # Momentum term when using SGD.
            'nesterov': False,                  # Enable Nesterov updates for SGD if desired.
        },
        'scheduler': {
            'name': 'cosine',                   # Scheduler type (cosine or onecycle).
            't_max': 15,                        # Cosine period, usually matches number of epochs.
            'max_lr': 2.5e-3,                   # Peak LR when using OneCycle (ignored otherwise).
            'pct_start': 0.3,                   # OneCycle warm-up proportion.
            'div_factor': 25.0,                 # Initial LR divider for OneCycle.
            'final_div_factor': 10000.0,        # Final LR divider for OneCycle cooldown.
        },
    },

    # Augmentation probabilities and ranges powering optional image-space regularisation.
    'augmentation': {
        'enable': True,                             # Master switch for train-time augmentation.
        'random_resized_crop_scale': (0.6, 1.0),    # Scale bounds for RandomResizedCrop.
        'random_resized_crop_ratio': (0.75, 1.33),  # Aspect-ratio bounds for RandomResizedCrop.
        'horizontal_flip_prob': 0.5,                # Chance of mirroring each training image.
        'perspective_prob': 0.3,                    # Probability of applying RandomPerspective.
        'perspective_distortion': 0.08,             # Distortion strength for perspective warps.
        'color_jitter': {
            'brightness': 0.25,                 # Range for brightness jittering.
            'contrast': 0.25,                   # Range for contrast jittering.
            'saturation': 0.2,                  # Range for saturation jittering.
            'hue': 0.04,                        # Range for hue jittering.
        },
        'blur_prob': 0.3,                       # Probability of applying Gaussian blur.
        'blur_sigma': (0.1, 2.0),               # Sigma range for the blur kernel.
        'random_erasing_prob': 0.25,            # Chance of dropping a random patch.
        'random_erasing_scale': (0.02, 0.2),    # Area range for erased patches.
        'random_erasing_ratio': (0.3, 3.3),     # Aspect-ratio range for erased patches.
    },

    # Global run options (seed for reproducibility, output root for artefacts).
    'seed': 42,
    'output_root': Path('project_outputs_classification'),
}


In [ ]:

# --- Experiment Initialisation Steps ---

# Step 1 — create the output root and lock the random seed.
CLASS_CFG['output_root'].mkdir(parents=True, exist_ok=True)
cls_set_seed(CLASS_CFG['seed'])

# Step 2 — derive device/run identifiers and ensure the run directory exists.
CLASS_DEVICE = cls_device()                              # Select GPU or CPU.
CLASS_RUN_ID = cls_timestamp()                           # Unique timestamp for this run.
CLASS_RUN_DIR = CLASS_CFG['output_root'] / CLASS_RUN_ID  # Concrete output folder per run.
CLASS_RUN_DIR.mkdir(parents=True, exist_ok=True)

# Step 3 — surface the resolved configuration and target paths.
print('Classification configuration:')

def _json_ready(obj):
    """Convert Paths, tuples, lists, and nested dicts into JSON-friendly primitives."""
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, tuple):
        return [_json_ready(v) for v in obj]
    if isinstance(obj, list):
        return [_json_ready(v) for v in obj]
    if isinstance(obj, dict):
        return {k: _json_ready(v) for k, v in obj.items()}
    return obj

print(json.dumps(_json_ready(CLASS_CFG), indent=2))
print('Device:', CLASS_DEVICE)
print('Output folder:', CLASS_RUN_DIR)


### A.3 Data Preparation Section — Transforms, Dataset Wrappers, And Loaders

In [ ]:
CLASS_NAMES = ['Real', 'Synthetic', 'Tampered']  # Label mapping reused by datasets/loaders/metrics.

# --- Transform Factory ---
# Builds torchvision transforms driven entirely by CLASS_CFG so augment knobs live in config.
def class_transforms(cfg: dict, *, is_train: bool) -> T.Compose:
    data_cfg = cfg['data']
    aug_cfg = cfg['augmentation']
    image_size = data_cfg['image_size']
    augment_enabled = aug_cfg.get('enable', False) and is_train
    ops = []

    if augment_enabled:
        scale_min, scale_max = aug_cfg.get('random_resized_crop_scale', (0.6, 1.0))  # Crop scale bounds.
        ratio_min, ratio_max = aug_cfg.get('random_resized_crop_ratio', (0.75, 1.33))  # Crop aspect ratio bounds.
        ops.append(T.RandomResizedCrop(image_size, scale=(scale_min, scale_max), ratio=(ratio_min, ratio_max)))

        hflip_prob = aug_cfg.get('horizontal_flip_prob', 0.0)  # Horizontal flip chance.
        if hflip_prob > 0:
            ops.append(T.RandomHorizontalFlip(p=hflip_prob))

        perspective_prob = aug_cfg.get('perspective_prob', 0.0)  # RandomPerspective frequency.
        if perspective_prob > 0:
            ops.append(
                T.RandomApply(
                    [T.RandomPerspective(distortion_scale=aug_cfg.get('perspective_distortion', 0.08))],
                    p=perspective_prob,
                )
            )

        cj = aug_cfg.get('color_jitter', {})
        if any(cj.get(key, 0.0) > 0 for key in ('brightness', 'contrast', 'saturation', 'hue')):
            ops.append(
                T.ColorJitter(
                    brightness=cj.get('brightness', 0.0),
                    contrast=cj.get('contrast', 0.0),
                    saturation=cj.get('saturation', 0.0),
                    hue=cj.get('hue', 0.0),
                )
            )

        blur_prob = aug_cfg.get('blur_prob', 0.0)
        if blur_prob > 0:
            sigma_min, sigma_max = aug_cfg.get('blur_sigma', (0.1, 2.0))
            ops.append(
                T.RandomApply(
                    [T.GaussianBlur(kernel_size=3, sigma=(sigma_min, sigma_max))],
                    p=blur_prob,
                )
            )
    else:
        ops.append(T.Resize((image_size, image_size), antialias=True))  # Deterministic resize for eval/test.

    # Always convert to float tensors and normalise to ImageNet statistics expected by the model.
    ops.extend([
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    if augment_enabled and aug_cfg.get('random_erasing_prob', 0.0) > 0:
        scale_min, scale_max = aug_cfg.get('random_erasing_scale', (0.02, 0.2))
        ratio_min, ratio_max = aug_cfg.get('random_erasing_ratio', (0.3, 3.3))
        ops.append(
            T.RandomErasing(
                p=aug_cfg['random_erasing_prob'],
                scale=(scale_min, scale_max),
                ratio=(ratio_min, ratio_max),
            )
        )

    return T.Compose(ops)


In [ ]:
# --- Dataset Wrapper ---
# Converts Hugging Face dataset rows into tensors + labels that honour the configured transform.
class ClassificationDataset(Dataset):
    def __init__(self, hf_dataset, cfg: dict, *, is_train: bool):
        self.dataset = hf_dataset
        self.transform = class_transforms(cfg, is_train=is_train)

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        record = self.dataset[idx]
        image = record['image'].convert('RGB')  # HF stores PIL images; convert before transforms.
        tensor = self.transform(image)
        label = torch.tensor(int(record['label']), dtype=torch.long)
        return {'image': tensor, 'label': label}


In [ ]:
# --- DataLoader Factory 
# Pulls bounded splits from SID_Set, applies ClassificationDataset, then instantiates configured DataLoaders.
def build_class_loaders(cfg: dict) -> tuple[DataLoader, DataLoader, DataLoader]:
    data_cfg = cfg['data']
    loader_cfg = cfg['loader']
    dataset_name = data_cfg['dataset_name']

    hf_train = load_dataset(dataset_name, split=f"train[:{data_cfg['train_samples']}]")
    hf_val = load_dataset(dataset_name, split=f"validation[:{data_cfg['val_samples']}]")
    hf_test = load_dataset(
        dataset_name,
        split=f"validation[{data_cfg['val_samples']}:{data_cfg['val_samples'] + data_cfg['test_samples']}]",
    )

    train_ds = ClassificationDataset(hf_train, cfg, is_train=True)
    val_ds = ClassificationDataset(hf_val, cfg, is_train=False)
    test_ds = ClassificationDataset(hf_test, cfg, is_train=False)

    pin_memory = loader_cfg.get('pin_memory', True)
    persistent_workers = loader_cfg.get('persistent_workers', False) and loader_cfg['num_workers'] > 0
    loader_kwargs = {
        'batch_size': loader_cfg['batch_size'],
        'num_workers': loader_cfg['num_workers'],
        'pin_memory': pin_memory,
    }
    if persistent_workers:
        loader_kwargs['persistent_workers'] = True
    if loader_cfg['num_workers'] > 0 and loader_cfg.get('prefetch_factor') is not None:
        loader_kwargs['prefetch_factor'] = loader_cfg['prefetch_factor']

    train_loader = DataLoader(
        train_ds,
        shuffle=loader_cfg.get('shuffle_train', True),
        **loader_kwargs,
    )
    val_loader = DataLoader(
        val_ds,
        shuffle=loader_cfg.get('shuffle_eval', False),
        **loader_kwargs,
    )
    test_loader = DataLoader(
        test_ds,
        shuffle=loader_cfg.get('shuffle_eval', False),
        **loader_kwargs,
    )

    return train_loader, val_loader, test_loader

train_loader_c, val_loader_c, test_loader_c = build_class_loaders(CLASS_CFG)
print('Classification dataloaders -> train:', len(train_loader_c), 'val:', len(val_loader_c), 'test:', len(test_loader_c))


### A.4 Visual Inspection Section — Sanity Checking Augmentation

The rest of Part A is unchanged as it already follows best practices.

In [ ]:
# Visual Inspection Section, sanity Checking Augmentation
batch_preview = next(iter(train_loader_c))
grid = vutils.make_grid(batch_preview['image'][:8], nrow=4, normalize=True)
plt.figure(figsize=(6, 6))
plt.imshow(np.transpose(grid.cpu().numpy(), (1, 2, 0)))
plt.title('Augmented training samples')
plt.axis('off')
plt.show()

### A.5 Model Definition Section — Compact Convolutional Architecture

In [ ]:
# Model Architecture
class Classifier(nn.Module):
    def __init__(self, base_width: int = 32, num_classes: int = 3):
        super().__init__()
        widths = [base_width, base_width * 2, base_width * 4]
        self.stem = self._block(3, widths[0])
        self.stage1 = self._block(widths[0], widths[1])
        self.stage2 = self._block(widths[1], widths[2])
        self.pool = nn.MaxPool2d(2)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(widths[2], widths[2]),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(widths[2], num_classes),
        )

    @staticmethod
    def _block(in_ch: int, out_ch: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.stem(x))
        x = self.pool(self.stage1(x))
        x = self.stage2(x)
        return self.head(x)

### A.6 Optimisation Setup Section — Loss Function, Optimiser, And Scheduler

In [ ]:
# Optimisation Setup Section — Loss Function, Optimiser, And Scheduler

training_cfg = CLASS_CFG['training']
optimizer_cfg = training_cfg['optimizer']

def build_optimizer(parameters, train_cfg: dict):
    name = train_cfg['optimizer'].get('name', 'adamw').lower()
    lr = train_cfg['learning_rate']
    opt_cfg = train_cfg['optimizer']
    weight_decay = opt_cfg.get('weight_decay', 0.0)
    betas = opt_cfg.get('betas', (0.9, 0.999))
    if name == 'adam':
        return optim.Adam(parameters, lr=lr, betas=betas, weight_decay=weight_decay)
    if name == 'sgd':
        return optim.SGD(
            parameters,
            lr=lr,
            momentum=opt_cfg.get('momentum', 0.9),
            weight_decay=weight_decay,
            nesterov=opt_cfg.get('nesterov', False),
        )
    return optim.AdamW(parameters, lr=lr, betas=betas, weight_decay=weight_decay)

def build_scheduler(optimizer, train_cfg: dict, steps_per_epoch: int):
    sched_cfg = train_cfg.get('scheduler', {})
    name = sched_cfg.get('name', 'none').lower()
    epochs = train_cfg['epochs']
    if name == 'cosine':
        t_max = sched_cfg.get('t_max', epochs)
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=t_max), 'epoch'
    if name == 'onecycle':
        max_lr = sched_cfg.get('max_lr', train_cfg['learning_rate'])
        scheduler = optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=max_lr,
            epochs=epochs,
            steps_per_epoch=max(steps_per_epoch, 1),
            pct_start=sched_cfg.get('pct_start', 0.3),
            div_factor=sched_cfg.get('div_factor', 25.0),
            final_div_factor=sched_cfg.get('final_div_factor', 10000.0),
        )
        return scheduler, 'batch'
    return None, 'epoch'

model_c = Classifier(num_classes=len(CLASS_NAMES)).to(CLASS_DEVICE)
criterion_c = nn.CrossEntropyLoss(label_smoothing=training_cfg['label_smoothing'])
optimizer_c = build_optimizer(model_c.parameters(), training_cfg)
amp_enabled = training_cfg.get('use_amp', True) and CLASS_DEVICE.type == 'cuda'
scaler_c = amp.GradScaler(enabled=amp_enabled)
scheduler_c, scheduler_step_mode = build_scheduler(optimizer_c, training_cfg, len(train_loader_c))
ema_c = Classifier(num_classes=len(CLASS_NAMES)).to(CLASS_DEVICE)
ema_c.load_state_dict(model_c.state_dict())
for param in ema_c.parameters():
    param.requires_grad_(False)


### A.7 Training Loop Section — Epoch Scheduling And EMA Tracking

In [ ]:
# Training Loop Section — Epoch Scheduling And EMA Tracking

training_cfg = CLASS_CFG['training']
epochs = training_cfg['epochs']
grad_clip_norm = training_cfg['grad_clip_norm']
ema_decay = training_cfg['ema_decay']
amp_enabled = training_cfg.get('use_amp', True) and CLASS_DEVICE.type == 'cuda'

history_c = defaultdict(list)
best_state_c = None
best_val_acc = -math.inf

for epoch in range(epochs):
    model_c.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch in train_loader_c:
        images = batch['image'].to(CLASS_DEVICE)
        labels = batch['label'].to(CLASS_DEVICE)

        optimizer_c.zero_grad(set_to_none=True)
        with amp.autocast(device_type=CLASS_DEVICE.type, enabled=amp_enabled):
            logits = model_c(images)
            loss = criterion_c(logits, labels)

        scaler_c.scale(loss).backward()

        if grad_clip_norm > 0:
            scaler_c.unscale_(optimizer_c)
            torch.nn.utils.clip_grad_norm_(model_c.parameters(), grad_clip_norm)

        scaler_c.step(optimizer_c)
        scaler_c.update()

        if ema_decay > 0:
            for ema_param, param in zip(ema_c.parameters(), model_c.parameters()):
                ema_param.data.mul_(ema_decay).add_(param.data, alpha=1 - ema_decay)

        if scheduler_c is not None and scheduler_step_mode == 'batch':
            scheduler_c.step()

        train_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += images.size(0)

    if scheduler_c is not None and scheduler_step_mode == 'epoch':
        scheduler_c.step()

    model_c.eval()
    ema_c.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch in val_loader_c:
            images = batch['image'].to(CLASS_DEVICE)
            labels = batch['label'].to(CLASS_DEVICE)
            logits = ema_c(images)
            loss = criterion_c(logits, labels)
            val_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += images.size(0)

    train_loss /= max(train_total, 1)
    train_acc = train_correct / max(train_total, 1)
    val_loss /= max(val_total, 1)
    val_acc = val_correct / max(val_total, 1)

    history_c['train_loss'].append(train_loss)
    history_c['train_acc'].append(train_acc)
    history_c['val_loss'].append(val_loss)
    history_c['val_acc'].append(val_acc)
    history_c['lr'].append(optimizer_c.param_groups[0]['lr'])

    print(f"[Classification] Epoch {epoch + 1:02d}/{epochs} train_loss={train_loss:.3f} train_acc={train_acc:.3f} val_loss={val_loss:.3f} val_acc={val_acc:.3f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_c = {k: v.clone() for k, v in ema_c.state_dict().items()}

if best_state_c is not None:
    ema_c.load_state_dict(best_state_c)


### A.8 Evaluation Section — Metrics, Visualisations, And Artefacts

In [ ]:
# Evaluation Section — Metrics, Visualisations, And Artefacts

model_path_c = CLASS_RUN_DIR / 'best_model.pth'
history_path_c = CLASS_RUN_DIR / 'history.json'
torch.save(ema_c.state_dict(), model_path_c)
with history_path_c.open('w') as fp:
    json.dump({k: v for k, v in history_c.items()}, fp, indent=2)
print('Saved classification model to', model_path_c)

ema_c.eval()
all_preds: list[int] = []
all_labels: list[int] = []

with torch.no_grad():
    for batch in test_loader_c:
        images = batch['image'].to(CLASS_DEVICE)
        labels = batch['label'].to(CLASS_DEVICE)
        logits = ema_c(images)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

report = classification_report(
    all_labels,
    all_preds,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
cm = confusion_matrix(all_labels, all_preds)
(CLASS_RUN_DIR / 'evaluation.json').write_text(
    json.dumps({'classification_report': report, 'confusion_matrix': cm.tolist()}, indent=2)
)
print('Classification macro F1:', report['macro avg']['f1-score'])

plt.figure(figsize=(8, 4))
plt.plot(history_c['train_loss'], label='Train')
plt.plot(history_c['val_loss'], label='Validation')
plt.title('Classification Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_c['train_acc'], label='Train')
plt.plot(history_c['val_acc'], label='Validation')
plt.title('Classification Accuracy Curves')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_c['lr'])
plt.title('Classification Learning Rate Schedule')
plt.xlabel('Epoch')
plt.ylabel('LR')
plt.show()

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.title('Classification Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

sample_batch = next(iter(test_loader_c))
images = sample_batch['image'][:6].to(CLASS_DEVICE)
labels = sample_batch['label'][:6]

with torch.no_grad():
    logits = ema_c(images)
    preds = logits.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for ax, img, pred, label in zip(axes.flatten(), images.cpu(), preds, labels):
    img_disp = (
        img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    )
    pred_idx = int(pred.item())
    label_idx = int(label.item())
    ax.imshow(np.transpose(img_disp.clamp(0, 1).numpy(), (1, 2, 0)))
    ax.set_title(f"Pred: {CLASS_NAMES[pred_idx]} | True: {CLASS_NAMES[label_idx]}")
    ax.axis('off')

plt.tight_layout()
plt.show()

print('Classification artefacts:')
print(''.join(sorted(p.name for p in CLASS_RUN_DIR.iterdir())))

## Part B — Segmentation Pipeline

The second half of the project walks through a U-Net style segmentation model that highlights manipulated regions. Each subsection isolates a single responsibility so you can understand how masks are prepared, how the network is constructed, and how results are measured without jumping between cells.

### B.1 Initialization Section — Imports, Reproducibility, And Device Helpers

In [ ]:
# ==================================================================================================
# Initialization Section — Imports, Reproducibility, And Device Helpers
# ==================================================================================================
# The segmentation walkthrough depends on PyTorch for tensor operations, torchvision for mask-aware
# transforms, Hugging Face Datasets for streaming SID_Set records, and matplotlib/seaborn for visual
# diagnostics. Keeping these imports together documents exactly what needs to be installed beforehand.
import json
import math
import random
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch import amp
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
# IMPROVEMENT: Added albumentations for a robust data augmentation pipeline.
import albumentations as A
from albumentations.pytorch import ToTensorV2

def seg_set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seg_timestamp() -> str:
    return datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')

def seg_device() -> torch.device:
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### B.2 Configuration Section — Experiment Hyperparameters And Run Directory

In [ ]:
# ==================================================================================================
# Configuration Section — Experiment Hyperparameters And Run Directory
# ==================================================================================================
SEG_CFG = {
    'dataset_name': 'saberzl/SID_Set',
    'image_size': 256,
    'train_samples': 220,
    'val_samples': 80,
    'test_samples': 80,
    'batch_size': 8,
    # IMPROVEMENT: Set num_workers > 0 to speed up data loading by using parallel processes.
    # Adjust based on the number of CPU cores available on your machine (e.g., 4, 8).
    'num_workers': 4,
    'learning_rate': 6e-4,
    'epochs': 18,
    'grad_clip_norm': 1.5,
    'ema_decay': 0.99,
    'use_cosine_schedule': True,
    'augment': True,
    'seed': 1337,
    'output_root': Path('project_outputs_segmentation'),
}

SEG_CFG['output_root'].mkdir(parents=True, exist_ok=True)
seg_set_seed(SEG_CFG['seed'])

SEG_DEVICE = seg_device()
SEG_RUN_ID = seg_timestamp()
SEG_RUN_DIR = SEG_CFG['output_root'] / SEG_RUN_ID
SEG_RUN_DIR.mkdir(parents=True, exist_ok=True)

print('Segmentation configuration:')
print(json.dumps({k: (str(v) if isinstance(v, Path) else v) for k, v in SEG_CFG.items()}, indent=2))
print('Device:', SEG_DEVICE)
print('Output folder:', SEG_RUN_DIR)

### B.3 Data Preparation Section — Augmentation And Dataset Utilities

The following section has been significantly refactored. The custom augmentation logic has been replaced with `Albumentations`, an industry-standard library for fast and reliable image augmentations. This simplifies the code, reduces potential bugs, and makes the augmentation pipeline easier to modify.

#### B.3.1 Dataset Filtering Utilities — Select Masked Tampered Frames

In [ ]:
_SEG_LABEL_CACHE: dict[tuple[str, int], list[int]] = {}

def seg_label_indices(hf_dataset, label_value: int) -> list[int]:
    """Cache indices for samples matching the requested label to minimise repeated filtering work."""
    fingerprint = getattr(hf_dataset, '_fingerprint', None)
    cache_key = (str(fingerprint), label_value)
    if cache_key not in _SEG_LABEL_CACHE:
        labels = hf_dataset['label']
        _SEG_LABEL_CACHE[cache_key] = [idx for idx, lbl in enumerate(labels) if lbl == label_value]
    return _SEG_LABEL_CACHE[cache_key]

def seg_filter(hf_dataset, max_samples):
    """Select tampered frames that provide masks, warning if fewer samples are available than requested."""
    candidate_indices = seg_label_indices(hf_dataset, label_value=2)
    picked_indices: list[int] = []
    for idx in candidate_indices:
        record = hf_dataset[idx]
        if record['mask'] is None:
            continue
        picked_indices.append(idx)
        if max_samples is not None and len(picked_indices) >= max_samples:
            break
    if max_samples is not None and len(picked_indices) < max_samples:
        print(f"[seg_filter] Requested {max_samples} samples but only found {len(picked_indices)} with masks.")
    return hf_dataset.select(picked_indices)

#### B.3.2 Dataset Wrapper with Albumentations

This new `SegDataset` class uses Albumentations to handle all transformations for both images and masks, ensuring they stay synchronized.

In [ ]:
def get_seg_transforms(image_size: int, augment: bool) -> A.Compose:
    """Builds an Albumentations pipeline for segmentation tasks."""
    if augment:
        return A.Compose([
            A.RandomResizedCrop(height=image_size, width=image_size, scale=(0.6, 1.0), ratio=(0.75, 1.33), p=1.0),
            A.HorizontalFlip(p=0.5),
            A.Perspective(scale=(0.0, 0.08), p=0.3),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.0, p=0.8),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(height=image_size, width=image_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])

class SegDataset(Dataset):
    """Return paired image and mask tensors from the filtered Hugging Face dataset using Albumentations."""
    def __init__(self, hf_dataset, image_size, augment=False):
        self.dataset = hf_dataset
        self.transform = get_seg_transforms(image_size, augment)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        record = self.dataset[idx]
        image = np.array(record['image'].convert('RGB'))
        mask = np.array(record['mask'].convert('L'))
        mask = (mask > 127).astype(np.float32) # Binarize mask

        transformed = self.transform(image=image, mask=mask)
        return {'image': transformed['image'], 'mask': transformed['mask'].unsqueeze(0)}


### B.4 Data Loader Section — Dataset Construction And Visual Sanity Check

The dataset splitting logic is corrected here to ensure a proper separation between validation and test sets. The validation set is now a hold-out from the training data, and the test set is derived from the dataset's original `validation` split.

#### B.4.1 Instantiate Dataset Splits And Loaders

In [ ]:
full_train_raw = seg_filter(load_dataset(SEG_CFG['dataset_name'], split='train', streaming=False), SEG_CFG['train_samples'] + SEG_CFG['val_samples'])
test_raw = seg_filter(load_dataset(SEG_CFG['dataset_name'], split='validation', streaming=False), SEG_CFG['test_samples'])

train_indices = range(SEG_CFG['train_samples'])
val_indices = range(SEG_CFG['train_samples'], SEG_CFG['train_samples'] + SEG_CFG['val_samples'])
train_raw = full_train_raw.select(train_indices)
val_raw = full_train_raw.select(val_indices)

train_ds_seg = SegDataset(train_raw, SEG_CFG['image_size'], augment=SEG_CFG['augment'])
val_ds_seg = SegDataset(val_raw, SEG_CFG['image_size'], augment=False)
test_ds_seg = SegDataset(test_raw, SEG_CFG['image_size'], augment=False)

train_loader_seg = DataLoader(train_ds_seg, batch_size=SEG_CFG['batch_size'], shuffle=True, num_workers=SEG_CFG['num_workers'])
val_loader_seg = DataLoader(val_ds_seg, batch_size=SEG_CFG['batch_size'], shuffle=False, num_workers=SEG_CFG['num_workers'])
test_loader_seg = DataLoader(test_ds_seg, batch_size=SEG_CFG['batch_size'], shuffle=False, num_workers=SEG_CFG['num_workers'])

print('Segmentation dataset sizes -> train:', len(train_raw), 'val:', len(val_raw), 'test:', len(test_raw))
print('Segmentation dataloaders -> train:', len(train_loader_seg), 'val:', len(val_loader_seg), 'test:', len(test_loader_seg))

#### B.4.2 Preview Augmented Samples And Masks

In [ ]:
preview_batch = next(iter(train_loader_seg))
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(4):
    img = preview_batch['image'][i]
    mask = preview_batch['mask'][i]
    img_disp = img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1) + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    axes[0, i].imshow(np.transpose(img_disp.clamp(0, 1).numpy(), (1, 2, 0)))
    axes[0, i].set_title('Augmented Image')
    axes[0, i].axis('off')
    axes[1, i].imshow(mask.squeeze(0).numpy(), cmap='gray')
    axes[1, i].set_title('Corresponding Mask')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

### B.5 Model And Optimisation Section — Architecture, Losses, And Training Tools

The rest of Part B is unchanged, as the model architecture, loss functions, and training loop are already well-designed.

#### B.5.1 U-Net Architecture Definition

In [ ]:
class SegConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, *, use_batchnorm: bool = True):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=not use_batchnorm),
            nn.BatchNorm2d(out_channels) if use_batchnorm else nn.Identity(),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=not use_batchnorm),
            nn.BatchNorm2d(out_channels) if use_batchnorm else nn.Identity(),
            nn.ReLU(inplace=True),
        ]
        self.block = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, base_width: int = 32):
        super().__init__()
        b = base_width
        self.enc1 = SegConvBlock(3, b)
        self.enc2 = SegConvBlock(b, b * 2)
        self.enc3 = SegConvBlock(b * 2, b * 4)
        self.enc4 = SegConvBlock(b * 4, b * 8)
        self.pool = nn.MaxPool2d(2)
        self.center = SegConvBlock(b * 8, b * 16)
        self.up4 = nn.ConvTranspose2d(b * 16, b * 8, kernel_size=2, stride=2)
        self.dec4 = SegConvBlock(b * 16, b * 8)
        self.up3 = nn.ConvTranspose2d(b * 8, b * 4, kernel_size=2, stride=2)
        self.dec3 = SegConvBlock(b * 8, b * 4)
        self.up2 = nn.ConvTranspose2d(b * 4, b * 2, kernel_size=2, stride=2)
        self.dec2 = SegConvBlock(b * 4, b * 2)
        self.up1 = nn.ConvTranspose2d(b * 2, b, kernel_size=2, stride=2)
        self.dec1 = SegConvBlock(b * 2, b)
        self.head = nn.Conv2d(b, 1, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool(enc1))
        enc3 = self.enc3(self.pool(enc2))
        enc4 = self.enc4(self.pool(enc3))
        center = self.center(self.pool(enc4))
        dec4 = self.dec4(torch.cat([self.up4(center), enc4], dim=1))
        dec3 = self.dec3(torch.cat([self.up3(dec4), enc3], dim=1))
        dec2 = self.dec2(torch.cat([self.up2(dec3), enc2], dim=1))
        dec1 = self.dec1(torch.cat([self.up1(dec2), enc1], dim=1))
        return self.head(dec1)

#### B.5.2 Loss And Metric Utilities

In [ ]:
def dice_loss(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    num = (probs * targets).sum(dim=(1, 2, 3))
    den = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return 1 - (2 * num + eps) / (den + eps)

def dice_coefficient(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()
    num = (preds * targets).sum(dim=(1, 2, 3))
    den = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return (2 * num + eps) / (den + eps)

def iou_coefficient(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - intersection
    return (intersection + eps) / (union + eps)

#### B.5.3 Optimisation Setup — Loss Function, Optimiser, Scheduler, EMA

In [ ]:
model_seg = UNet(base_width=32).to(SEG_DEVICE)
criterion_bce_seg = nn.BCEWithLogitsLoss()
optimizer_seg = optim.AdamW(model_seg.parameters(), lr=SEG_CFG['learning_rate'], weight_decay=1e-4)
scaler_seg = amp.GradScaler(enabled=(SEG_DEVICE.type == 'cuda'))
scheduler_seg = (
    optim.lr_scheduler.CosineAnnealingLR(optimizer_seg, T_max=SEG_CFG['epochs'])
    if SEG_CFG['use_cosine_schedule']
    else None
)
ema_seg = UNet(base_width=32).to(SEG_DEVICE)
ema_seg.load_state_dict(model_seg.state_dict())
for param in ema_seg.parameters():
    param.requires_grad_(False)

### B.6 Training Loop Section — Dice-Focused Optimisation

In [ ]:
# ==================================================================================================
# Training Loop Section — Dice-Focused Optimisation
# ==================================================================================================
history_seg = defaultdict(list)
best_state_seg = None
best_val_dice = -math.inf

for epoch in range(SEG_CFG['epochs']):
    model_seg.train()
    train_loss = 0.0
    train_dice = 0.0
    batches = 0

    for batch in train_loader_seg:
        images = batch['image'].to(SEG_DEVICE)
        masks = batch['mask'].to(SEG_DEVICE)

        optimizer_seg.zero_grad(set_to_none=True)
        with amp.autocast(device_type='cuda', enabled=(SEG_DEVICE.type == 'cuda')):
            logits = model_seg(images)
            bce = criterion_bce_seg(logits, masks)
            dice = dice_loss(logits, masks).mean()
            loss = 0.5 * bce + 0.5 * dice

        scaler_seg.scale(loss).backward()
        if SEG_CFG['grad_clip_norm'] > 0:
            scaler_seg.unscale_(optimizer_seg)
            torch.nn.utils.clip_grad_norm_(model_seg.parameters(), SEG_CFG['grad_clip_norm'])
        scaler_seg.step(optimizer_seg)
        scaler_seg.update()

        if SEG_CFG['ema_decay'] > 0:
            for ema_param, param in zip(ema_seg.parameters(), model_seg.parameters()):
                ema_param.data.mul_(SEG_CFG['ema_decay']).add_(param.data, alpha=1 - SEG_CFG['ema_decay'])

        train_loss += loss.item()
        train_dice += dice_coefficient(logits, masks).mean().item()
        batches += 1

    if scheduler_seg is not None:
        scheduler_seg.step()

    model_seg.eval()
    ema_seg.eval()
    val_loss = 0.0
    val_dice = 0.0
    val_iou = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_loader_seg:
            images = batch['image'].to(SEG_DEVICE)
            masks = batch['mask'].to(SEG_DEVICE)
            logits = ema_seg(images)
            bce = criterion_bce_seg(logits, masks)
            dice = dice_loss(logits, masks).mean()
            loss = 0.5 * bce + 0.5 * dice
            val_loss += loss.item()
            val_dice += dice_coefficient(logits, masks).mean().item()
            val_iou += iou_coefficient(logits, masks).mean().item()
            val_batches += 1

    train_loss /= max(batches, 1)
    train_dice /= max(batches, 1)
    val_loss /= max(val_batches, 1)
    val_dice /= max(val_batches, 1)
    val_iou /= max(val_batches, 1)

    history_seg['train_loss'].append(train_loss)
    history_seg['train_dice'].append(train_dice)
    history_seg['val_loss'].append(val_loss)
    history_seg['val_dice'].append(val_dice)
    history_seg['val_iou'].append(val_iou)
    history_seg['lr'].append(optimizer_seg.param_groups[0]['lr'])

    print(
        f"[Segmentation] Epoch {epoch + 1:02d}/{SEG_CFG['epochs']} "
        f"train_loss={train_loss:.3f} train_dice={train_dice:.3f} "
        f"val_loss={val_loss:.3f} val_dice={val_dice:.3f} val_iou={val_iou:.3f}"
    )

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_state_seg = {k: v.clone() for k, v in ema_seg.state_dict().items()}

if best_state_seg is not None:
    ema_seg.load_state_dict(best_state_seg)

### B.7 Evaluation Section — Metrics, Visualisations, And Artefacts

In [ ]:
# ==================================================================================================
# Evaluation Section — Metrics, Visualisations, And Artefacts
# ==================================================================================================
model_path_seg = SEG_RUN_DIR / 'best_model.pth'
history_path_seg = SEG_RUN_DIR / 'history.json'
torch.save(ema_seg.state_dict(), model_path_seg)
with history_path_seg.open('w') as fp:
    json.dump({k: v for k, v in history_seg.items()}, fp, indent=2)
print('Saved segmentation model to', model_path_seg)

segmentation_dice_scores: list[float] = []
segmentation_iou_scores: list[float] = []
qualitative_examples: list[tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = []
ema_seg.eval()
with torch.no_grad():
    for batch in test_loader_seg:
        images = batch['image'].to(SEG_DEVICE)
        masks = batch['mask'].to(SEG_DEVICE)
        logits = ema_seg(images)
        segmentation_dice_scores.extend(dice_coefficient(logits, masks).cpu().tolist())
        segmentation_iou_scores.extend(iou_coefficient(logits, masks).cpu().tolist())
        preds = (torch.sigmoid(logits) > 0.5).float()
        if len(qualitative_examples) < 9:
            for img, mask, pred in zip(images, masks, preds):
                if len(qualitative_examples) < 9:
                    qualitative_examples.append((img.cpu(), mask.cpu(), pred.cpu()))

(SEG_RUN_DIR / 'evaluation.json').write_text(
    json.dumps(
        {
            'dice_scores_mean': float(np.mean(segmentation_dice_scores)),
            'iou_scores_mean': float(np.mean(segmentation_iou_scores)),
        },
        indent=2,
    )
)
print(f"Segmentation dice {np.mean(segmentation_dice_scores):.3f} ± {np.std(segmentation_dice_scores):.3f}")
print(f"Segmentation IoU  {np.mean(segmentation_iou_scores):.3f} ± {np.std(segmentation_iou_scores):.3f}")

plt.figure(figsize=(8, 4))
plt.plot(history_seg['train_loss'], label='Train')
plt.plot(history_seg['val_loss'], label='Validation')
plt.title('Segmentation Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_seg['train_dice'], label='Train Dice')
plt.plot(history_seg['val_dice'], label='Validation Dice')
plt.title('Segmentation Dice Curves')
plt.xlabel('Epoch')
plt.ylabel('Dice')
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_seg['val_iou'], label='Validation IoU')
plt.title('Segmentation IoU Curve')
plt.xlabel('Epoch')
plt.ylabel('IoU')
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_seg['lr'])
plt.title('Segmentation Learning Rate Schedule')
plt.xlabel('Epoch')
plt.ylabel('LR')
plt.show()

rows = min(len(qualitative_examples), 3)
fig, axes = plt.subplots(rows, 3, figsize=(9, 3 * rows))
if rows == 1:
    axes = np.expand_dims(axes, axis=0)
for row, (img, mask, pred) in enumerate(qualitative_examples[:rows]):
    img_disp = (
        img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    )
    axes[row, 0].imshow(np.transpose(img_disp.clamp(0, 1).numpy(), (1, 2, 0)))
    axes[row, 0].set_title('Image')
    axes[row, 0].axis('off')
    axes[row, 1].imshow(mask.squeeze(0).numpy(), cmap='gray')
    axes[row, 1].set_title('Ground Truth Mask')
    axes[row, 1].axis('off')
    axes[row, 2].imshow(pred.squeeze(0).numpy(), cmap='gray')
    axes[row, 2].set_title('Predicted Mask')
    axes[row, 2].axis('off')
plt.tight_layout()
plt.show()

print('Segmentation artefacts:')
print(''.join(sorted(p.name for p in SEG_RUN_DIR.iterdir())))